In [ ]:
import matplotlib.pyplot as plt
from scipy.fft import rfft, rfftfreq, irfft
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
def Xi1D_direct_periodic(x):
    N, Npix = x.shape
    xi = np.empty((N, Npix), dtype=float)
    for lag in range(Npix):
        xi[:, lag] = np.mean(x * np.roll(x, -lag, axis=1), axis=1)
    return xi

In [ ]:
output_dir = Path("./plots")
output_dir.mkdir(parents=True, exist_ok=True)

# Parameters
N = 64
Npix = 1024
L_box = 100
n_pairs = 3
sigma_spike = L_box/1000

delta_v_values = np.linspace(L_box/100, L_box/1.5, 64)
for delta_v_0 in delta_v_values:

    # Derived quantities
    dv = L_box / Npix
    v = np.arange(Npix) * dv
    x = np.zeros((N, Npix))

    rng = np.random.default_rng(137)
    for s in range(N):
        for _ in range(n_pairs):
            c1 = rng.uniform(0, L_box)
            c2 = (c1 + delta_v_0) % L_box
            d1 = ((v - c1 + L_box/2) % L_box) - L_box/2
            d2 = ((v - c2 + L_box/2) % L_box) - L_box/2
            x[s] -= np.exp(-0.5 * (d1 / sigma_spike)**2)
            x[s] -= np.exp(-0.5 * (d2 / sigma_spike)**2)

    x_mean = np.mean(x, axis=1, keepdims=True)
    delta_x = x / x_mean - 1.0

    delta_tag = f"{delta_v_0 / L_box:.5f}".replace(".", "p")

    idx = rng.choice(x.shape[0], size=3, replace=False)
    fig, axes = plt.subplots(figsize=(16, 6), ncols=1, nrows=2, sharex=True)
    for i in idx:
        axes[0].plot(v / L_box, x[i], lw=2.0, alpha=0.7)
    for i in idx:
        axes[1].plot(v / L_box, delta_x[i], lw=2.0, alpha=0.7)
    axes[0].set_ylabel(r"$x(v)$", fontsize=18)
    axes[0].tick_params(axis='both', labelsize=18)
    axes[0].set_ylim(-1.5, 0.0)
    axes[1].set_xlabel(r"$v / L_\mathrm{box}$", fontsize=18)
    axes[1].set_ylabel(r"$\delta_x(v)$", fontsize=18)
    axes[1].tick_params(axis='both', labelsize=18)
    axes[1].set_ylim(-1.0, 70)
    plt.tight_layout()
    fig.savefig(output_dir / f"fields_delta_{delta_tag}.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

    # P1D from rfft
    fk = rfft(delta_x, axis=1)
    k = 2.0 * np.pi * rfftfreq(Npix, d=dv)
    Pk = (dv / Npix) * (fk * fk.conj()).real   # shape: (N, Nfreq)
    Pk_mean = Pk.mean(axis=0)

    # 2PCF from irfft
    xi = irfft(Pk, n=Npix, axis=1) / dv        # shape: (N, Npix)
    xi_mean = xi.mean(axis=0)
    # Center the lag axis for plotting
    lags = (np.arange(Npix) - Npix // 2) * dv
    xi_mean_plot = np.fft.fftshift(xi_mean)

    # 2PCF from direct computation
    xi_per = Xi1D_direct_periodic(delta_x)
    xi_per_mean = xi_per.mean(axis=0)
    lags_per = np.arange(delta_x.shape[1]) * dv

    # Normalized coordinates
    k_norm = k * L_box / (2.0 * np.pi)     # mode number n
    lags_norm = lags / L_box
    lags_per_norm = lags_per / L_box
    delta_norm = delta_v_0 / L_box

    # Plot P1D
    fig, ax = plt.subplots(figsize=(16, 4))
    ax.plot(k_norm[1:], Pk_mean[1:], lw=3.0, color='black')
    ax.set_xlabel(r"$kL/(2\pi)$", fontsize=18)
    ax.set_ylabel(r"$P_{\rm 1D}(k)$", fontsize=18)
    ax.tick_params(axis='both', labelsize=18)
    plt.tight_layout()
    fig.savefig(output_dir / f"P1D_delta_{delta_tag}.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

    # Plot Xi1D
    fig, ax = plt.subplots(figsize=(16, 4))
    ax.plot(lags_norm, xi_mean_plot, lw=3.0, color='k', label=r"iRFFT")
    ax.plot(lags_per_norm, xi_per_mean, lw=1.2, color='red', label=r"direct calculation")
    ax.axvline(0.5, color='k', lw=1.5, alpha=0.9, ls=':')
    ax.axvline(delta_norm, color='k', lw=1.5, alpha=0.9, ls='--')
    ymin, ymax = ax.get_ylim()
    ytext = ymin + 0.92 * (ymax - ymin)
    ax.text(0.5, ytext, r"$L_\mathrm{box}/2$", ha='center', va='top', fontsize=16)
    ax.text(delta_norm, ytext, r"$\Delta v_0/L_\mathrm{box}$", ha='center', va='top', fontsize=16)
    ax.set_xlabel(r"$\Delta v / L_\mathrm{box}$", fontsize=18)
    ax.set_ylabel(r"$\xi_{\rm 1D}(\Delta v)$", fontsize=18)
    ax.legend(fontsize=16, )
    ax.tick_params(axis='both', labelsize=18)
    plt.tight_layout()
    fig.savefig(output_dir / f"Xi1D_delta_{delta_tag}.png", dpi=200, bbox_inches="tight")
    plt.close(fig)